In [2]:
from google.colab import drive; drive.mount('/content/drive')
import sys, json, shutil, importlib, numpy as np, pandas as pd, torch
from sklearn.isotonic import IsotonicRegression
from transformers import AutoTokenizer, AutoModelForSequenceClassification
CODE = "/content/drive/MyDrive/esci/code"; sys.path.insert(0, CODE)
import metrics, cross_encoder as C, train_cross_encoder as T
for m in (metrics, C, T): importlib.reload(m)
BASE = "/content/drive/MyDrive/esci"

# ---- validation data only (test set is never used again)
df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")
qs = pd.read_parquet(f"{BASE}/dataset_v2.3/esci_electronics_v2.3_queries.parquet")[["query_id", "rule", "scope_group"]]
val = df[df.part == "val"].merge(qs, on="query_id", how="left").reset_index(drop=True); del df
CAT = {"mobile_computing": "Mobile & Computing", "components_accessories": "Components & Accessories",
       "home_electronics": "Home Electronics", "imaging": "Cameras & Imaging", "audio": "Audio",
       "computer_peripherals": "Computer Peripherals", "gaming": "Gaming", "smart_home": "Smart Home"}
val["category"] = np.where(val["rule"] == "A", val["scope_group"].map(CAT), None)
val["q"] = [C.query_to_text(q, category=c) for q, c in zip(val["query"], val["category"])]
val["text"] = [C.product_to_text(C.esci_row_to_product(r), "full_bullets") for r in val.to_dict("records")]
val["label"] = val["gain"].round(2).map({1.0: "E", 0.1: "S", 0.01: "C", 0.0: "I"})
assert val.label.notna().all()

# ---- (1) deployment model: augplus seed with the highest val NDCG on normal formats
L = pd.read_csv(f"{BASE}/results/s9e_augplus_val_log.csv"); L = L[L["style"].isin(C.STYLES)]
ep = L.groupby(["run", "epoch"])["ndcg@10"].mean().reset_index()
bestep = ep.loc[ep.groupby("run")["ndcg@10"].idxmax()].sort_values("ndcg@10", ascending=False)
print(bestep.to_string(index=False))
RUN, EPOCH = bestep.iloc[0]["run"], int(bestep.iloc[0]["epoch"])
path = f"{BASE}/models/{RUN}/epoch_{EPOCH}"; print("\nDEPLOYMENT MODEL:", path)

# ---- (2) scores on validation
tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path).to("cuda").eval()
logit = np.asarray(T.score_pairs(model, tok, val["q"].tolist(), val["text"].tolist(), 256, 128), dtype=float)
val["p"] = 1 / (1 + np.exp(-logit)) if (logit.min() < 0 or logit.max() > 1) else logit
print("\nScore by label (validation):")
print(val.groupby("label")["p"].describe()[["count", "mean", "25%", "50%", "75%"]].reindex(list("ESCI")).round(3).to_string())

# ---- (3) calibrated P(Exact) with isotonic regression, then thresholds (rule fixed before running)
iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip")
iso.fit(val["p"].values, (val["label"] == "E").astype(float).values)
val["p_exact"] = iso.predict(val["p"].values)

def lowest_score_with(prob):              # smallest raw score whose calibrated P(Exact) >= prob
    ok = val.loc[val.p_exact >= prob, "p"]
    return float(ok.min()) if len(ok) else None
t_great, t_good = lowest_score_with(0.90), lowest_score_with(0.50)
print(f"\nt_great = {t_great:.3f}  (P(Exact) >= 0.90)   t_good = {t_good:.3f}  (P(Exact) >= 0.50)")

val["bin"] = pd.qcut(val["p"], 10, duplicates="drop")
print("\nCalibration (score bin -> real share of Exact, mean calibrated P):")
print(val.groupby("bin", observed=True).agg(n=("p", "size"), share_E=("label", lambda x: (x == "E").mean()),
                                            p_exact=("p_exact", "mean")).round(3).to_string())

val["match"] = np.where(val.p >= t_great, "great", np.where(val.p >= t_good, "good", "fair"))
print("\nLabel mix inside each match label (each row sums to 1):")
print(pd.crosstab(val["match"], val["label"], normalize="index").reindex(["great", "good", "fair"])[list("ESCI")].round(3).to_string())
print("share of products per match label:", val["match"].value_counts(normalize=True).round(3).to_dict())

# ---- (4) low_confidence rule on validation
words = val["product_title"].fillna("").str.split().str.len()
empty = lambda c: val[c].fillna("").astype(str).str.strip().eq("")
val["low_conf"] = (words < 6) & empty("product_brand") & empty("product_bullet_point")
print(f"\nlow_confidence rows on ESCI val: {val.low_conf.mean():.3%}")
print(val[val.label == "E"].groupby("low_conf")["p"].agg(["mean", "count"]).round(3).to_string())

# ---- (5) save the deployment package (model + config with calibration points)
OUT = f"{BASE}/models/m4_final"; shutil.copytree(path, OUT, dirs_exist_ok=True)
cfg = {"source_run": RUN, "epoch": EPOCH, "query_input": "q_focused_cat", "product_text": "full_bullets",
       "max_length": 256, "score": "sigmoid(logit)", "calibration": "isotonic P(Exact) fitted on ESCI val",
       "cal_x": iso.X_thresholds_.tolist(), "cal_y": iso.y_thresholds_.tolist(),
       "great_if_p_exact_ge": 0.90, "good_if_p_exact_ge": 0.50, "t_great": t_great, "t_good": t_good,
       "low_conf_rule": "title_words<6 and no brand and no features"}
json.dump(cfg, open(f"{OUT}/m4_config.json", "w"), indent=2)
val[["query_id", "product_id", "label", "p", "p_exact", "match", "low_conf"]].to_csv(
    f"{BASE}/results/s12_val_calibration.csv", index=False)
print("\nsaved:", OUT)
print({k: v for k, v in cfg.items() if k not in ("cal_x", "cal_y")}, "| calibration points:", len(cfg["cal_x"]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                                  run  epoch  ndcg@10
minilm_l6_augplus__q_focused_cat__s43      2   0.9279
     minilm_l6_augplus__q_focused_cat      3   0.9277
minilm_l6_augplus__q_focused_cat__s44      3   0.9268

DEPLOYMENT MODEL: /content/drive/MyDrive/esci/models/minilm_l6_augplus__q_focused_cat__s43/epoch_2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Score by label (validation):
        count   mean    25%    50%    75%
label                                    
E      8099.0  0.838  0.789  0.971  0.994
S      2509.0  0.362  0.131  0.237  0.546
C       586.0  0.346  0.126  0.226  0.497
I       547.0  0.287  0.075  0.162  0.431

t_great = 0.953  (P(Exact) >= 0.90)   t_good = 0.424  (P(Exact) >= 0.50)

Calibration (score bin -> real share of Exact, mean calibrated P):
                                    n  share_E  p_exact
bin                                                    
(2.0000000000000052e-05, 0.131]  1176    0.130    0.129
(0.131, 0.24]                    1173    0.247    0.244
(0.24, 0.452]                    1174    0.411    0.413
(0.452, 0.709]                   1175    0.655    0.654
(0.709, 0.882]                   1174    0.756    0.753
(0.882, 0.956]                   1175    0.841    0.844
(0.956, 0.983]                   1180    0.937    0.937
(0.983, 0.992]                   1172    0.971    0.968
(0.992, 0.996]  